# 10 - Quantification Data Foundation

This notebook verifies the restored NIST SD302 research inputs before any automated ridge-count or minutiae algorithm is developed. It uses aggregate summaries only; subject-linked rows and biometric images remain in ignored private folders.

The goals are to:

- confirm archive provenance and image linkage,
- measure expert core, delta, and minutiae annotation coverage,
- confirm that expert subtype labels are linked back to subjects,
- define which quantification analyses are currently defensible.

In [ ]:
from pathlib import Path
import json

import pandas as pd

## Private restoration paths

The restoration utility has already verified the NIST checksums and generated private linkage tables. We locate those outputs relative to the repository so the notebook works from either the project root or the `notebooks` directory.

In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESTORE_DIR = PROJECT_ROOT / "data" / "processed" / "sd302_2026_restoration"
RESULTS_DIR = PROJECT_ROOT / "results" / "quantification_foundation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Restoration directory:", RESTORE_DIR)

## Load the restoration audit

The manifest records source hashes and aggregate recovery counts. The feature inventory contains one row per IRR record, while the subtype linkage contains only the 637 accepted expert labels.

In [ ]:
manifest = json.loads((RESTORE_DIR / "restoration_manifest.json").read_text())
inventory = pd.read_csv(RESTORE_DIR / "irr_feature_inventory.csv", dtype={"subject_id": str, "finger_position": str})
subtype_links = pd.read_csv(RESTORE_DIR / "subtype_subject_linkage.csv", dtype={"subject_id": str, "finger_position": str})

manifest["source"], inventory.shape, subtype_links.shape

## Integrity gates

These assertions stop the notebook if a source checksum failed, an image link is missing, or a subtype record did not recover its subject identifier.

In [ ]:
assert all(source["verified"] for source in manifest["sources"])
assert len(inventory) == 2380
assert inventory["image_found"].all()
assert inventory["subject_id"].nunique() == 200
assert len(subtype_links) == 637
assert subtype_links["subject_id"].notna().all()

print("All restoration integrity gates passed.")

## Examiner feature coverage

A field being present means that the expert EFS transaction contains that feature family. It does not mean that every fingerprint should contain the same number of features.

In [ ]:
coverage = pd.DataFrame([
    {"feature": "Core", "records": inventory["core_count"].gt(0).sum(), "points": inventory["core_count"].sum()},
    {"feature": "Delta", "records": inventory["delta_count"].gt(0).sum(), "points": inventory["delta_count"].sum()},
    {"feature": "Minutia", "records": inventory["minutiae_count"].gt(0).sum(), "points": inventory["minutiae_count"].sum()},
])
coverage["record_coverage_pct"] = (100 * coverage["records"] / len(inventory)).round(2)
coverage

In [ ]:
broad_counts = (
    inventory["broad_class"]
    .value_counts()
    .rename_axis("broad_class")
    .reset_index(name="images")
)
broad_counts

## Restored subtype grouping

Images from the same person are not independent observations. We therefore count both images and unique subjects for every subtype before selecting a grouped validation design.

In [ ]:
subtype_coverage = (
    subtype_links.groupby("confirmed_subtype")
    .agg(images=("review_id", "size"), subjects=("subject_id", "nunique"))
    .reset_index()
    .sort_values("confirmed_subtype")
)
subtype_coverage

In [ ]:
subtype_links["family"] = subtype_links["confirmed_subtype"].str.endswith("arch").map({True: "arch", False: "whorl"})
family_coverage = (
    subtype_links.groupby("family")
    .agg(images=("review_id", "size"), subjects=("subject_id", "nunique"))
    .reset_index()
)
family_coverage

## Quantification decisions

The restored EFS coordinates provide expert reference data for evaluating automated core, delta, and minutiae extraction. Ridge count still requires a documented core-to-delta tracing algorithm. Total ridge count and individual pattern intensity additionally require a ten-finger aggregation protocol.

In [ ]:
decisions = pd.DataFrame([
    {"parameter": "Minutiae", "current_basis": "Expert 9.331 coordinates and types", "next_validation": "Compare automated detections with examiner points"},
    {"parameter": "Finger ridge count", "current_basis": "Expert cores and deltas plus source PNGs", "next_validation": "Implement and manually audit core-to-delta crossings"},
    {"parameter": "Total ridge count", "current_basis": "Restored subject and finger positions", "next_validation": "Freeze ten-finger and whorl-count convention"},
    {"parameter": "Pattern intensity", "current_basis": "Expert broad pattern codes", "next_validation": "Freeze individual and cohort aggregation formulas"},
])
decisions

## Save aggregate evidence

Only non-identifying aggregate tables are written to `results/`. Subject IDs, record keys, coordinates, and image paths remain in ignored private files.

In [ ]:
coverage.to_csv(RESULTS_DIR / "expert_feature_coverage.csv", index=False)
broad_counts.to_csv(RESULTS_DIR / "restored_broad_class_counts.csv", index=False)
subtype_coverage.to_csv(RESULTS_DIR / "grouped_subtype_coverage.csv", index=False)
family_coverage.to_csv(RESULTS_DIR / "grouped_family_coverage.csv", index=False)
decisions.to_csv(RESULTS_DIR / "quantification_decisions.csv", index=False)

print("Saved aggregate tables to:", RESULTS_DIR)

## Conclusion

The raw-data limitation is resolved for the current study. All accepted subtype labels have recovered subject identifiers, and the refreshed SD302g annotations provide examiner reference points for quantification. The next analytical stage should rerun subtype model selection with subject-grouped folds before implementing and validating automated minutiae and ridge-count measurements.